# 📊 NOTEBOOK 1 : PRÉPARATION DES DONNÉES

**Objectif :** Charger, nettoyer et préparer les données Kaggle pour le ML.

**Fichiers utilisés :**
- `ratings_small.csv` (100k ratings)
- `movies_metadata.csv` (45k films)
- `keywords.csv` (mots-clés)

**Output :** `../data/processed/movies_enriched.csv`

In [1]:
# Imports
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports OK")

✅ Imports OK


## 1. CHARGER LES DONNÉES

In [2]:
# Chemin vers tes fichiers (CHANGE SI NÉCESSAIRE)
DATA_PATH = '../data/raw/'  # ou './archive/' si tu es dans notebooks/

print("📥 Chargement des données...\n")

# 1. Ratings
ratings = pd.read_csv(DATA_PATH + 'ratings_small.csv')
print(f"✅ Ratings : {len(ratings):,} lignes, {ratings['userId'].nunique():,} users, {ratings['movieId'].nunique():,} films")

# 2. Movies
movies = pd.read_csv(DATA_PATH + 'movies_metadata.csv', low_memory=False)
print(f"✅ Movies : {len(movies):,} lignes")

# 3. Keywords
keywords = pd.read_csv(DATA_PATH + 'keywords.csv')
print(f"✅ Keywords : {len(keywords):,} lignes")

print("\n📊 Aperçu des ratings :")
print(ratings.head())

📥 Chargement des données...

✅ Ratings : 100,004 lignes, 671 users, 9,066 films
✅ Movies : 45,466 lignes
✅ Keywords : 46,419 lignes

📊 Aperçu des ratings :
   userId  movieId  rating   timestamp
0       1       31     2.5  1260759144
1       1     1029     3.0  1260759179
2       1     1061     3.0  1260759182
3       1     1129     2.0  1260759185
4       1     1172     4.0  1260759205


## 2. NETTOYER MOVIES_METADATA

In [3]:
print("🧹 Nettoyage de movies_metadata...\n")

# Nettoyer l'id (parfois lu comme string)
movies['id'] = pd.to_numeric(movies['id'], errors='coerce')
movies = movies.dropna(subset=['id'])
movies['id'] = movies['id'].astype(int)

# Supprimer les films sans titre ni overview
movies = movies.dropna(subset=['title', 'overview'])

print(f"✅ Après nettoyage : {len(movies):,} films")
print(f"\n📋 Colonnes disponibles : {list(movies.columns)}")

🧹 Nettoyage de movies_metadata...

✅ Après nettoyage : 44,506 films

📋 Colonnes disponibles : ['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count']


## 3. PARSER LES GENRES (JSON → STRING)

In [4]:
print("🎬 Parsing des genres...\n")

def parse_genres(genres_str):
    """Convertit '[{"id": 16, "name": "Animation"}]' en 'Animation'"""
    try:
        genres = ast.literal_eval(genres_str)
        return ' '.join([g['name'] for g in genres])
    except:
        return ''

movies['genres_clean'] = movies['genres'].apply(parse_genres)

print("Exemple de parsing :")
print(movies[['title', 'genres', 'genres_clean']].head(3))
print(f"\n✅ Genres parsés : {movies['genres_clean'].notna().sum():,} films")

🎬 Parsing des genres...

Exemple de parsing :
              title                                             genres  \
0         Toy Story  [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   
1           Jumanji  [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   
2  Grumpier Old Men  [{'id': 10749, 'name': 'Romance'}, {'id': 35, ...   

               genres_clean  
0   Animation Comedy Family  
1  Adventure Fantasy Family  
2            Romance Comedy  

✅ Genres parsés : 44,506 films


## 4. PARSER LES KEYWORDS

In [5]:
print("🔑 Parsing des keywords...\n")

def parse_keywords(keywords_str):
    """Extrait les top 10 keywords"""
    try:
        kw = ast.literal_eval(keywords_str)
        return ' '.join([k['name'] for k in kw[:10]])
    except:
        return ''

keywords['keywords_clean'] = keywords['keywords'].apply(parse_keywords)

print("Exemple :")
print(keywords[['id', 'keywords_clean']].head(3))
print(f"\n✅ Keywords parsés : {len(keywords):,} films")

🔑 Parsing des keywords...

Exemple :
      id                                     keywords_clean
0    862  jealousy toy boy friendship friends rivalry bo...
1   8844  board game disappearance based on children's b...
2  15602   fishing best friend duringcreditsstinger old men

✅ Keywords parsés : 46,419 films


## 5. FUSIONNER LES DATASETS

In [6]:
print("🔗 Fusion des datasets...\n")

# Fusionner movies + keywords
movies_full = movies.merge(
    keywords[['id', 'keywords_clean']], 
    on='id', 
    how='left'
)

print(f"✅ Fusion OK : {len(movies_full):,} films")
print(f"\n📊 Films avec keywords : {movies_full['keywords_clean'].notna().sum():,}")

🔗 Fusion des datasets...

✅ Fusion OK : 45,484 films

📊 Films avec keywords : 45,483


## 6. CRÉER LA COLONNE METADATA (POUR TF-IDF)

In [7]:
print("📝 Création de la colonne metadata...\n")

# Combine genres + overview + keywords
movies_full['metadata'] = (
    movies_full['genres_clean'].fillna('') + ' ' +
    movies_full['overview'].fillna('') + ' ' +
    movies_full['keywords_clean'].fillna('')
)

# Vérifie
print("Exemple de metadata :")
sample = movies_full[movies_full['title'] == 'Toy Story'].iloc[0]
print(f"\nTitre : {sample['title']}")
print(f"Metadata : {sample['metadata'][:200]}...")

print(f"\n✅ Metadata créée pour {len(movies_full):,} films")

📝 Création de la colonne metadata...

Exemple de metadata :

Titre : Toy Story
Metadata : Animation Comedy Family Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against B...

✅ Metadata créée pour 45,484 films


## 7. FILTRER LES FILMS AVEC RATINGS

In [8]:
print("🎯 Filtrage des films notés...\n")

# Garder seulement les films qui ont des ratings
movies_with_ratings = movies_full[movies_full['id'].isin(ratings['movieId'])].copy()

print(f"✅ Films avec ratings : {len(movies_with_ratings):,}")
print(f"Coverage : {len(movies_with_ratings) / ratings['movieId'].nunique() * 100:.1f}%")

🎯 Filtrage des films notés...

✅ Films avec ratings : 2,821
Coverage : 31.1%


## 8. SÉLECTIONNER LES COLONNES UTILES

In [9]:
print("📦 Sélection des colonnes finales...\n")

# Colonnes à garder
cols_finales = [
    'id', 'title', 'genres_clean', 'overview', 
    'keywords_clean', 'metadata', 'vote_average', 
    'vote_count', 'release_date'
]

movies_final = movies_with_ratings[cols_finales].copy()
movies_final.rename(columns={'id': 'movieId'}, inplace=True)

print(f"✅ Dataset final : {len(movies_final):,} films × {len(movies_final.columns)} colonnes")
print(f"\nColonnes : {list(movies_final.columns)}")

📦 Sélection des colonnes finales...

✅ Dataset final : 2,821 films × 9 colonnes

Colonnes : ['movieId', 'title', 'genres_clean', 'overview', 'keywords_clean', 'metadata', 'vote_average', 'vote_count', 'release_date']


## 9. VÉRIFICATIONS FINALES

In [10]:
print("🔍 Vérifications finales...\n")

print("📊 Statistiques :")
print(f"  Ratings : {len(ratings):,} lignes")
print(f"  Films : {len(movies_final):,} lignes")
print(f"  Users : {ratings['userId'].nunique():,}")
print(f"  Films notés : {ratings['movieId'].nunique():,}")

print("\n📋 Valeurs manquantes dans movies_final :")
print(movies_final.isnull().sum())

print("\n✅ Aperçu final :")
print(movies_final.head())

🔍 Vérifications finales...

📊 Statistiques :
  Ratings : 100,004 lignes
  Films : 2,821 lignes
  Users : 671
  Films notés : 9,066

📋 Valeurs manquantes dans movies_final :
movieId           0
title             0
genres_clean      0
overview          0
keywords_clean    0
metadata          0
vote_average      0
vote_count        0
release_date      1
dtype: int64

✅ Aperçu final :
    movieId                  title                 genres_clean  \
5       949                   Heat  Action Crime Drama Thriller   
9       710              GoldenEye    Adventure Action Thriller   
14     1408       Cutthroat Island             Action Adventure   
15      524                 Casino                  Drama Crime   
16     4584  Sense and Sensibility                Drama Romance   

                                             overview  \
5   Obsessive master thief, Neil McCauley leads a ...   
9   James Bond must unmask the mysterious head of ...   
14  Morgan Adams and her slave, William Sh

## 10. SAUVEGARDER

In [11]:
import os

# Créer le dossier si nécessaire
os.makedirs('../data/processed', exist_ok=True)

# Sauvegarder
movies_final.to_csv('../data/processed/movies_enriched.csv', index=False)
ratings.to_csv('../data/processed/ratings_clean.csv', index=False)

print("💾 Sauvegarde...")
print("✅ ../data/processed/movies_enriched.csv")
print("✅ ../data/processed/ratings_clean.csv")
print("\n🎉 DONNÉES PRÊTES POUR LE ML !")

💾 Sauvegarde...
✅ ../data/processed/movies_enriched.csv
✅ ../data/processed/ratings_clean.csv

🎉 DONNÉES PRÊTES POUR LE ML !


---

# ✅ RÉSUMÉ

**Fichiers générés :**
- `movies_enriched.csv` : Films avec metadata (genres + overview + keywords)
- `ratings_clean.csv` : Ratings nettoyés

**Prochaine étape : Notebook 2 — Entraîner le SVD**